In [1]:
import ray
from ray.train import ScalingConfig, RunConfig, Checkpoint
from ray.train.torch import TorchTrainer
import torch
import torch.nn as nn
import torch.nn.functional as F
import tempfile
import os
import numpy as np
import pandas as pd

# Re-train (Update) Recommender using Ray Train

We want to update and finetune our recommender model periodically (perhaps very frequently) using recent interaction data from our users.

Let's see how to train this model using Ray Data and Ray Train.

We'll start with the basic PyTorch code and local training, to see the original structure.

In [2]:
class TwoTower(nn.Module):
    def __init__(self, num_users: int, num_items: int, dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)

        # optional: small projection MLPs (kept minimal)
        self.user_proj = nn.Identity()
        self.item_proj = nn.Identity()

    def encode_users(self, user_ids: torch.LongTensor) -> torch.Tensor:
        u = self.user_proj(self.user_emb(user_ids))
        return F.normalize(u, dim=-1)

    def encode_items(self, item_ids: torch.LongTensor) -> torch.Tensor:
        v = self.item_proj(self.item_emb(item_ids))
        return F.normalize(v, dim=-1)

    def forward(self, user_ids: torch.LongTensor, pos_item_ids: torch.LongTensor):
        """
        Returns logits matrix [B,B] where diagonal is the positive pair and
        off-diagonals are in-batch negatives.
        """
        u = self.encode_users(user_ids)         # [B, D]
        v = self.encode_items(pos_item_ids)     # [B, D]
        logits = u @ v.t()                      # [B, B]
        return logits

In [3]:
def train_loop():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    num_users, num_items, dim = 1000, 1000, 64
    model = TwoTower(num_users, num_items, dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    # fake training data: (user, positive_item)
    for step in range(200):
        B = 256
        user_ids = torch.randint(0, num_users, (B,), device=device)
        pos_item_ids = torch.randint(0, num_items, (B,), device=device)

        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)     # [B,B]
        targets = torch.arange(logits.size(0), device=logits.device)  # diagonal
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
    
        if step % 50 == 0:
            print(f"step={step} loss={loss.item():.4f}")

In [4]:
train_loop()

step=0 loss=5.5546
step=50 loss=5.5516
step=100 loss=5.5517
step=150 loss=5.5463


## Minimal port onto Ray Train

The following is a very minimal port onto Ray Train for distributed Torch DDP training.

There are a number of elements we'll want to improve, but the core conversion from Torch to Ray Train + Torch DDP is straightforward.

Note that `prepare_model` manages device detection and model/data movement to/from devices as well performs the `DDP(model)` wrapping that we would code for Torch distributed.

> this training loop will be deployed into a training process in each worker

In [5]:
def train_loop_ray_minimal():
    num_users, num_items, dim = 1000, 1000, 64
    model = TwoTower(num_users, num_items, dim)  # .to(device) <------ device detection and management implicit
    
    model = ray.train.torch.prepare_model(model) # <------ wrap model, manage devices

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    # fake training data: (user, positive_item)
    for step in range(200):
        B = 256
        user_ids = torch.randint(0, num_users, (B,)) # <------ device detection and management implicit
        pos_item_ids = torch.randint(0, num_items, (B,)) # <------ device detection and management implicit

        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)     # [B,B]
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
        
        if step % 50 == 0:
            print(f"step={step} loss={loss.item():.4f}")

To orchestrate the training, we use `TorchTrainer` with a minimal config pointing to the training loop code, `ScalingConfig`, and a symmetric read/write storage path.

In [6]:
trainer = TorchTrainer(train_loop_ray_minimal, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'))

result = trainer.fit()

result

2026-08-24 14:54:35,747	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 100.125.138.24:6379...
2026-08-24 14:54:35,775	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-iy1mz6uapeim3bm6iel8iqvdpm.i.anyscaleuserdata.com 
2026-08-24 14:54:35,779	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_49df1b8f760e45fe52525885144143893f591889.zip' (0.69MiB) to Ray cluster...
2026-08-24 14:54:35,782	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_49df1b8f760e45fe52525885144143893f591889.zip'.
/home/ray/anaconda3/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(TrainController pid=68185) Requesting resources: {'CPU': 1} * 2
(Trai

(RayTrainWorker pid=31296, ip=100.93.121.85) [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(RayTrainWorker pid=31296, ip=100.93.121.85) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=68185) Started training worker group of size 2: 
(TrainController pid=68185) - (ip=100.93.121.85, pid=31296) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=68185) - (ip=100.93.121.85, pid=31295) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=68185) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=31296, ip=100.93.121.85) Moving model to device: cpu
(RayTrainWorker pid=31296, ip=100.93.121.85) Wrapping provided model in DistributedDataParallel.
(RayTrainWorker pid=31295, ip=100.93.121.85) step=0 loss=5.5544
(RayTrainWorker pid=31296, ip=100.93.121.85) step=0 loss=5.5640
(RayTrainWorker pid=31295, ip=100.93.121.85) step=50 loss=5.5611
(RayTrainWorker pid=31296, ip=100.93.121.85) step=50 loss=5.5505
(RayTrainWorker pid=31295, ip=100.93.121.85) step=100 loss=5.5540
(RayTrainWorker pid=31296, ip=100.93.121.85) step=100 loss=5.

Result(metrics=None, checkpoint=None, error=None, path='/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-35', metrics_dataframe=None, best_checkpoints=[], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x768becc46cb0>)

We'll re-write this training loop to add two improvements:
* parametrizing some values using a `config` dict that is supplied from our orchestration code, supporting better separation of concerns
* add model checkpointing and reporting of stats through Ray Train APIs
  * note that in some frameworks (e.g., Lightning) this checkpointing and reporting does not need to be coded by the user

In [7]:
def train_loop_ray_checkpoint_stats_and_config(config):
    
    num_users, num_items, dim = config['num_users'], config['num_items'], config['dim']
    model = TwoTower(num_users, num_items, dim) # values from config
    
    model = ray.train.torch.prepare_model(model)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    # fake training data: (user, positive_item)
    for step in range(200):
        B = 256
        user_ids = torch.randint(0, num_users, (B,))
        pos_item_ids = torch.randint(0, num_items, (B,))

        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)     # [B,B]
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
        
        if step % 50 == 0:
            with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
                checkpoint = None

                # In standard DDP training, where the model is the same across all ranks,
                # only the global rank 0 worker needs to save and report the checkpoint
                if ray.train.get_context().get_world_rank() == 0:
                    torch.save(
                        model.module.state_dict(),  # NOTE: Unwrap the model.
                        os.path.join(temp_checkpoint_dir, "model.pt"),
                    )
                    checkpoint = Checkpoint.from_directory(temp_checkpoint_dir)

                ray.train.report({'loss': loss.item()}, checkpoint=checkpoint)    

(TrainController pid=68185) [State Transition] SHUTTING_DOWN -> FINISHED.


The `config` values are supplied from the driver/orchestrator in the `TorchTrainer` constructor

In [8]:
trainer = TorchTrainer(train_loop_ray_checkpoint_stats_and_config, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64 })

result = trainer.fit()

result

(TrainController pid=68538) Requesting resources: {'CPU': 1} * 2
(TrainController pid=68538) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=68538) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2


(RayTrainWorker pid=31435, ip=100.93.121.85) [Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(RayTrainWorker pid=31434, ip=100.93.121.85) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=68538) Started training worker group of size 2: 
(TrainController pid=68538) - (ip=100.93.121.85, pid=31434) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=68538) - (ip=100.93.121.85, pid=31435) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=68538) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=31434, ip=100.93.121.85) Moving model to device: cpu
(RayTrainWorker pid=31434, ip=100.93.121.85) Wrapping provided model in DistributedDataParallel.
(RayTrainWorker pid=31435, ip=100.93.121.85) Reporting training result 1: TrainingReport(checkpoint=None, metrics={'loss': 5.557195663452148}, validation=False)
(RayTrainWorker pid=31434, ip=100.93.121.85) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-49/checkpoint_2026-08-24_14-54-58.671828)
(RayTrainWorker

Result(metrics={'loss': 5.5511474609375}, checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-49/checkpoint_2026-08-24_14-55-01.908048), error=None, path='/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-49', metrics_dataframe=       loss
0  5.535723
1  5.564927
2  5.540335
3  5.551147, best_checkpoints=[(Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-49/checkpoint_2026-08-24_14-54-58.671828), {'loss': 5.5357232093811035}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-49/checkpoint_2026-08-24_14-54-59.084788), {'loss': 5.564927101135254}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-49/checkpoint_2026-08-24_14-54-59.867558), {'loss': 5.540335178375244}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-54-49/checkpoint_2026-08-24_14-55-01.908048), {'loss': 5.5511474609375})], _stora

## Integrate Ray Data pipeline for supplying training data

We would like to leverage the Ray Data + Ray Train integration in order to
* simplify, parallelize, and accelerate feature pre-processing for training
* take advantage of Ray's resource scheduling to operate on different hardware for data processing vs. training
* optimize and accelerate the delivery of training data batches into the training workers

We'll do this in two steps:

1. Create a Ray Data pipeline (Dataset) that represents the featurized data we want to train on
2. Adjust our training worker code to consume batches of data from that pipeline

In [9]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True, file_extensions=['.ndjson'])

ds.take_batch(3)

(TrainController pid=68538) [State Transition] SHUTTING_DOWN -> FINISHED.
/home/ray/anaconda3/lib/python3.11/site-packages/ray/anyscale/data/api/read_api.py:586: UserWarning: orjson provides the fastest `read_json` implementation, but it’s not installed. Falling back to pandas. To use orjson, run: `pip install orjson`.
  warnings.warn(
2026-08-24 14:55:09,205	INFO logging.py:416 -- Registered dataset logger for dataset dataset_36_0
2026-08-24 14:55:09,236	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_36_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:55:09,236	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_36_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=3]
2026-08-24 14:55:09,240	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 28.0% of available memory (26.9GiB out of 96.0GiB total). F

{'id': array(['779e23ec-4714-4ba3-bc3a-2c8487cde669',
        '10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745',
        '7034dd99-ceb3-474d-a0ba-5beaf122273f'], dtype=object),
 'first_name': array(['Lucas', 'Emily', 'William'], dtype=object),
 'last_name': array(['Rodriguez', 'Taylor', 'Martinez'], dtype=object),
 'email': array(['lucas.rodriguez8648@hotmail.com', 'emily.taylor7287@icloud.com',
        'william.martinez7222@yahoo.com'], dtype=object),
 'last_20_positive_item_interactions': array([list([624, 208, 730, 714, 897, 86, 964, 267, 574, 260, 880, 600, 645, 882, 135, 273, 657, 203, 123, 974]),
        list([6, 612, 536, 705, 647, 704, 615, 73, 930, 350, 68, 324, 48, 446, 373, 160, 129, 418, 846, 318]),
        list([74, 875, 87, 859, 271, 184, 704, 616, 812, 547, 719, 549, 624, 26, 255, 889, 994, 326, 809, 653])],
       dtype=object)}

We'll implement a miniature version of our Database Facade and we'll add a `__call__` method to let us use it in a Dataset pipeline.

> Production Note: in order to improve separation of concerns, various patterns can split existing logic (such as database access code) away from the Ray Data specific code (such as the batch formats or `__call__`). These techniques include inheritance, factory patterns, or custom decorators.

In [10]:
class DatabaseFacade():
    def __init__(self, users):
        self.users = pd.read_json(users, lines=True)
        
    def users_for_ids(self, ids):
        return self.users[self.users['id'].isin(ids)]

    def __call__(self, batch):
        batch['user_indices'] = self.users_for_ids(batch['id']).index.values
        return batch

In [11]:
ds.select_columns(['id', 'last_20_positive_item_interactions']) \
    .map_batches(DatabaseFacade, fn_constructor_args=['/mnt/cluster_storage/ecom/users.ndjson']) \
    .take_batch(3)

2026-08-24 14:55:11,969	INFO logging.py:416 -- Registered dataset logger for dataset dataset_39_0
2026-08-24 14:55:11,975	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_39_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:55:11,975	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_39_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[Project->MapBatches(DatabaseFacade)] -> LimitOperator[limit=3]
{"asctime":"2026-08-24 14:55:11,999","levelname":"E","message":"Actor with class name: 'MapWorker(Project->MapBatches(DatabaseFacade))' and ID: '8bb17d0ef531691c7c8b290603000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.","filename":"core_worker.cc","li

{'id': array(['779e23ec-4714-4ba3-bc3a-2c8487cde669',
        '10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745',
        '7034dd99-ceb3-474d-a0ba-5beaf122273f'], dtype=object),
 'last_20_positive_item_interactions': array([array([624, 208, 730, 714, 897,  86, 964, 267, 574, 260, 880, 600, 645,
               882, 135, 273, 657, 203, 123, 974])                             ,
        array([  6, 612, 536, 705, 647, 704, 615,  73, 930, 350,  68, 324,  48,
               446, 373, 160, 129, 418, 846, 318])                             ,
        array([ 74, 875,  87, 859, 271, 184, 704, 616, 812, 547, 719, 549, 624,
                26, 255, 889, 994, 326, 809, 653])                             ],
       dtype=object),
 'user_indices': array([0, 1, 2])}

Our model expects a batch of inputs where each input is a user-product pair, so we want to explode each user and the user's 20 interactions into 20 pairs.

In [12]:
def explode_interactions(batch):
    batch_size = len(batch['id'])
    interactions = np.concatenate(batch['last_20_positive_item_interactions'])
    user_indices = np.concatenate([np.repeat(index, 20) for index in batch['user_indices']])
    return { 'user_indices' : user_indices, 'interactions' : interactions }

In [13]:
training_data_preprocessing_pipeline = ds.select_columns(['id', 'last_20_positive_item_interactions']) \
    .map_batches(DatabaseFacade, fn_constructor_args=['/mnt/cluster_storage/ecom/users.ndjson']) \
    .map_batches(explode_interactions)

training_data_preprocessing_pipeline.take(40)

2026-08-24 14:55:14,642	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-08-24 14:55:14,645	INFO logging.py:416 -- Registered dataset logger for dataset dataset_43_0
2026-08-24 14:55:14,651	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_43_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:55:14,651	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_43_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[Project->MapBatches(DatabaseFacade)] -> TaskPoolMapOperator[MapBatches(explode_interactions)] -> LimitOperator[limit=40]
2026-08-24 14:55:14,797	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_43_0 =======
2026-08-24 14:55:14,797	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:55:14,799	INFO logging_progress.py:227 -- 

[{'user_indices': 0, 'interactions': 624},
 {'user_indices': 0, 'interactions': 208},
 {'user_indices': 0, 'interactions': 730},
 {'user_indices': 0, 'interactions': 714},
 {'user_indices': 0, 'interactions': 897},
 {'user_indices': 0, 'interactions': 86},
 {'user_indices': 0, 'interactions': 964},
 {'user_indices': 0, 'interactions': 267},
 {'user_indices': 0, 'interactions': 574},
 {'user_indices': 0, 'interactions': 260},
 {'user_indices': 0, 'interactions': 880},
 {'user_indices': 0, 'interactions': 600},
 {'user_indices': 0, 'interactions': 645},
 {'user_indices': 0, 'interactions': 882},
 {'user_indices': 0, 'interactions': 135},
 {'user_indices': 0, 'interactions': 273},
 {'user_indices': 0, 'interactions': 657},
 {'user_indices': 0, 'interactions': 203},
 {'user_indices': 0, 'interactions': 123},
 {'user_indices': 0, 'interactions': 974},
 {'user_indices': 1, 'interactions': 6},
 {'user_indices': 1, 'interactions': 612},
 {'user_indices': 1, 'interactions': 536},
 {'user_indice

At this point, we have a data pipeline that produces the shape and flavor of data on which we want to train.

__Integrating the Dataset(s) with Ray Train code__

We'll implement two sets of changes:
1. consume batches of data within the train worker code
2. supply `Dataset`(s) (pipelines) to the Ray Train orchestrator, in the `Trainer` constructor

At a more detailed level, step 1 will...
* shard the dataset so that each worker gets a unique slice of the data for training
* create an iterator that produces batches of data for training, with options to
  * specify batch size
  * control data format and data type
  * prefetch for extra buffering
  * shuffle
* typically yield batches in the form of Python dicts, which we will index into to obtain NumPy or Torch tensors

In [14]:
def train_loop_ray_data(config):
    from ray.train import get_dataset_shard

    model = TwoTower(config['num_users'], config['num_items'], config['dim'])  # get values from config   
    model = ray.train.torch.prepare_model(model)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_sh = get_dataset_shard("train") # <--- get shard of data stream for the present worker
    training = train_sh.iter_torch_batches(batch_size=1000) # <--- get iterator of torch batches

    for batch in training:
        user_ids = batch['user_indices']
        pos_item_ids = batch['interactions']
    
        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
        
        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            checkpoint = None
            if ray.train.get_context().get_world_rank() == 0:
                torch.save(
                    model.module.state_dict(),
                    os.path.join(temp_checkpoint_dir, "model.pt"),
                )
                checkpoint = Checkpoint.from_directory(temp_checkpoint_dir)

            ray.train.report({'loss': loss.item()}, checkpoint=checkpoint)    

In [15]:
trainer = TorchTrainer(train_loop_ray_data, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64 },
                       datasets={'train' : training_data_preprocessing_pipeline}) # <--- supply training data pipeline

result = trainer.fit()

result

(TrainController pid=68922) Requesting resources: {'CPU': 1} * 2
(TrainController pid=68922) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=68922) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2


(RayTrainWorker pid=31813, ip=100.93.121.85) [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(RayTrainWorker pid=31813, ip=100.93.121.85) Setting up process group for: env:// [rank=0, world_size=2]
(RayTrainWorker pid=31813, ip=100.93.121.85) Moving model to device: cpu
(RayTrainWorker pid=31813, ip=100.93.121.85) Wrapping provided model in DistributedDataParallel.
(TrainController pid=68922) Started training worker group of size 2: 
(TrainController pid=68922) - (ip=100.93.121.85, pid=31813) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=68922) - (ip=100.93.121.85, pid=31812) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=68922) [State Transition] SCHEDULING -> RUNNING.
(SplitCoordinator pid=69168) Registered dataset logger for dataset train_44_0
(SplitCoordinator pid=69168) Starting execution of Dataset train_44_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
(SplitCoordinator pid=69168) Execution plan of Dataset train_44_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles

(pid=69168) Running Dataset train_44_0.: 0.00 row [00:00, ? row/s]

(pid=69168) - ListFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=69168) - MapBatches(explode_interactions):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=69168) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(RayTrainWorker pid=31812, ip=100.93.121.85) [Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(SplitCoordinator pid=69168) ✔️  Dataset train_44_0 execution finished in 2.57 seconds
(RayTrainWorker pid=31812, ip=100.93.121.85) Reporting training result 1: TrainingReport(checkpoint=None, metrics={'loss': 6.91538667678833}, validation=False)
(RayTrainWorker pid=31812, ip=100.93.121.85) Reporting training result 2: TrainingReport(checkpoint=None, metrics={'loss': 6.913564682006836}, validation=False)
(RayTrainWorker pid=31813, ip=100.93.121.85) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17/checkpoint_2026-08-24_14-55-31.207104)
(RayTrainWorker pid=31813, ip=100.93.121.85) Reporting training result 1: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17/checkpoint_2026-08-24_14-55-31.207104), metrics={'loss': 6.916690349578857}, validation=False)
(RayTrainWorker pid=31812, ip=100.93.121.85) Reporting training result 3: TrainingReport(checkpoint

Result(metrics={'loss': 6.91628360748291}, checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17/checkpoint_2026-08-24_14-55-45.972099), error=None, path='/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17', metrics_dataframe=       loss
0  6.916690
1  6.919503
2  6.914708
3  6.914819
4  6.919720
5  6.915651
6  6.913460
7  6.910348
8  6.910343
9  6.916284, best_checkpoints=[(Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17/checkpoint_2026-08-24_14-55-31.207104), {'loss': 6.916690349578857}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17/checkpoint_2026-08-24_14-55-31.454535), {'loss': 6.919503211975098}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17/checkpoint_2026-08-24_14-55-31.640156), {'loss': 6.914708137512207}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-17/che

Now that we can feed batches to our training, we can refactor a bit to produce an epoch/batch pattern, with reporting and checkpointing at each epoch.

In [16]:
def train_loop_ray_data_epochs(config):
    from ray.train import get_dataset_shard

    model = TwoTower(config['num_users'], config['num_items'], config['dim'])  # get values from config   
    model = ray.train.torch.prepare_model(model)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_sh = get_dataset_shard("train")
    
    global_batch_size = config['global_batch_size']
    per_worker_batch_size = global_batch_size // ray.train.get_context().get_world_size()
    training = train_sh.iter_torch_batches(batch_size=per_worker_batch_size) # <--- calculate local batch size if desired

    for epoch in range(config['num_epochs']):
        
        for batch in training:
            
            user_ids = batch['user_indices']
            pos_item_ids = batch['interactions']

            opt.zero_grad()

            logits = model(user_ids, pos_item_ids)
            targets = torch.arange(logits.size(0), device=logits.device)
            loss = F.cross_entropy(logits, targets)

            loss.backward()
            opt.step()

        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            checkpoint = None
            if ray.train.get_context().get_world_rank() == 0:
                torch.save(
                    model.module.state_dict(),  # NOTE: Unwrap the model.
                    os.path.join(temp_checkpoint_dir, "model.pt"),
                )
                checkpoint = Checkpoint.from_directory(temp_checkpoint_dir)

            ray.train.report({'loss': loss.item()}, checkpoint=checkpoint)    

(TrainController pid=68922) [State Transition] SHUTTING_DOWN -> FINISHED.


In [17]:
trainer = TorchTrainer(train_loop_ray_data_epochs, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64,
                                          'num_epochs' : 5,
                                          'global_batch_size' : 1024 },
                       datasets={'train' : training_data_preprocessing_pipeline})

result = trainer.fit()

result

(TrainController pid=69475) Requesting resources: {'CPU': 1} * 2
(TrainController pid=69475) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=69475) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2
(RayTrainWorker pid=32161, ip=100.93.121.85) Setting up process group for: env:// [rank=0, world_size=2]
(RayTrainWorker pid=32161, ip=100.93.121.85) Moving model to device: cpu
(RayTrainWorker pid=32161, ip=100.93.121.85) Wrapping provided model in DistributedDataParallel.
(TrainController pid=69475) Started training worker group of size 2: 
(TrainController pid=69475) - (ip=100.93.121.85, pid=32161) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=69475) - (ip=100.93.121.85, pid=32160) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=69475) [State Transition] SCHEDULING -> RUNNING.


(RayTrainWorker pid=32161, ip=100.93.121.85) [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
(raylet) WARNING: 4 PYTHON worker processes have been started on node: f987380542f8cbfb6253e55ae77bf5ade2c020d1e77df80460d6d8da with address: 100.125.138.24, which is 4x the maximum expected startup concurrency (1). This could be a result of using a large number of actors, tasks blocked in ray.get() calls, or tasks with fractional CPU requests (e.g., num_cpus=0.1) allowing high concurrency. See https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds.


(SplitCoordinator pid=69754) Registered dataset logger for dataset train_46_0
(SplitCoordinator pid=69754) Starting execution of Dataset train_46_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
(SplitCoordinator pid=69754) Execution plan of Dataset train_46_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[Project->MapBatches(DatabaseFacade)] -> TaskPoolMapOperator[MapBatches(explode_interactions)] -> OutputSplitter[split(2, equal=True)]


(pid=69754) Running Dataset train_46_0.: 0.00 row [00:00, ? row/s]

(pid=69754) - ListFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=69754) - Project->MapBatches(DatabaseFacade):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=69754) - MapBatches(explode_interactions):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=69754) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=69754) Registered dataset logger for dataset train_46_4
(SplitCoordinator pid=69754) Starting execution of Dataset train_46_4. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
(SplitCoordinator pid=69754) Execution plan of Dataset train_46_4: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[Project->MapBatches(DatabaseFacade)] -> TaskPoolMapOperator[MapBatches(explode_interactions)] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=69754) ✔️  Dataset train_46_4 execution finished in 2.29 seconds
(RayTrainWorker pid=32160, ip=100.93.121.85) Reporting training result 5: TrainingReport(checkpoint=None, metrics={'loss': 5.59744930267334}, validation=False)
(RayTrainWorker pid=32161, ip=100.93.121.85) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-53/checkpoint_2026-08-24_14-56-17.329019)

Result(metrics={'loss': 5.592108249664307}, checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-53/checkpoint_2026-08-24_14-56-17.329019), error=None, path='/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-53', metrics_dataframe=       loss
0  5.601235
1  5.598967
2  5.596664
3  5.594379
4  5.592108, best_checkpoints=[(Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-53/checkpoint_2026-08-24_14-56-06.798128), {'loss': 5.6012349128723145}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-53/checkpoint_2026-08-24_14-56-09.482770), {'loss': 5.598966598510742}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-53/checkpoint_2026-08-24_14-56-12.106201), {'loss': 5.5966644287109375}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-55-53/checkpoint_2026-08-24_14-56-14.711463), {'loss': 5.594379425

Note that in the multi-epoch training example, the Dataset is performing streaming execution -- including retrieving the data from a source -- for every epoch.

In many cases, that is the desired behavior. E.g., 
* the data may be so large that streaming and reprocessing represents a small cost compared to the cost of eliminating that work (e.g., storage, less optimal movement, etc.)
* or it may be necessary to perform some random, unique, or time-dependent computation (e.g., random data augmentation) producing new data values for each training epoch

However, in other cases, it may be possible to compute the dataset once and cache it in a place which is more local to the training cluster. In the latter case, we can compute the dataset and store it across the Ray Object Store (distributed memory but, in the case of a large dataset, likely spilling to a distributed disk cache), by calling `materialize` on the dataset.

In [18]:
cached_materialized_data = training_data_preprocessing_pipeline.materialize()

trainer = TorchTrainer(train_loop_ray_data_epochs, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64,
                                          'num_epochs' : 5,
                                          'global_batch_size' : 1024 },
                       datasets={'train' : cached_materialized_data})

result = trainer.fit()

result

(TrainController pid=69475) [State Transition] SHUTTING_DOWN -> FINISHED.
2026-08-24 14:56:20,537	INFO logging.py:416 -- Registered dataset logger for dataset dataset_48_0
2026-08-24 14:56:20,543	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_48_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:56:20,544	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_48_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[Project->MapBatches(DatabaseFacade)] -> TaskPoolMapOperator[MapBatches(explode_interactions)]
2026-08-24 14:56:20,693	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_48_0 =======
2026-08-24 14:56:20,694	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:56:20,694	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store (pending: 1 CPU)
2026-08-24 14:56:20,695

(RayTrainWorker pid=32802, ip=100.93.121.85) [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(RayTrainWorker pid=32802, ip=100.93.121.85) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=70048) Started training worker group of size 2: 
(TrainController pid=70048) - (ip=100.93.121.85, pid=32802) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=70048) - (ip=100.93.121.85, pid=32803) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=70048) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=32802, ip=100.93.121.85) Moving model to device: cpu
(RayTrainWorker pid=32802, ip=100.93.121.85) Wrapping provided model in DistributedDataParallel.
(SplitCoordinator pid=70340) Registered dataset logger for dataset train_50_0
(SplitCoordinator pid=70340) Starting execution of Dataset train_50_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
(SplitCoordinator pid=70340) Execution plan of Dataset train_50_0: InputDataBuffer[Input] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=7034

(pid=70340) Running Dataset train_50_0.: 0.00 row [00:00, ? row/s]

(pid=70340) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=70340) Running Dataset train_50_4.: 0.00 row [00:00, ? row/s]

(pid=70340) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(RayTrainWorker pid=32803, ip=100.93.121.85) Reporting training result 5: TrainingReport(checkpoint=None, metrics={'loss': 5.613339900970459}, validation=False)
(RayTrainWorker pid=32802, ip=100.93.121.85) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-39.415962)
(RayTrainWorker pid=32802, ip=100.93.121.85) Reporting training result 5: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-39.415962), metrics={'loss': 5.588497161865234}, validation=False)
(TrainController pid=70048) [State Transition] RUNNING -> SHUTTING_DOWN.


Result(metrics={'loss': 5.588497161865234}, checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-39.415962), error=None, path='/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22', metrics_dataframe=       loss
0  5.597490
1  5.595254
2  5.592984
3  5.590733
4  5.588497, best_checkpoints=[(Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-34.393332), {'loss': 5.597490310668945}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-34.835152), {'loss': 5.595253944396973}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-35.276685), {'loss': 5.592984199523926}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-37.319094), {'loss': 5.59073257446

(TrainController pid=70048) [State Transition] SHUTTING_DOWN -> FINISHED.


Note the difference in the Ray Data logging output when training from the materialized dataset.

Now we can wrap this training program in a Ray or Anyscale Job, and schedule it to regularly consume new data, train or finetune an existing model, and produce new model checkpoints in a known location.